# StrokeGuard AI — Phase 3: Data Preprocessing

Goal: clean the data and prepare it for model training, without leaking information from the test set into the training process.

**Key rule for this whole notebook: split first, clean second.**

In [4]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import SMOTE

os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)


## 1. Load the Data

In [5]:
df = pd.read_csv('../data/healthcare-dataset-stroke-data.csv')
print('Shape:', df.shape)
df.head()


Shape: (5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 2. Drop Columns / Rows We Don't Need

- `id` is just a row identifier, not medical information — drop it.
- There's exactly one row with `gender = 'Other'` in this dataset. With only 1 example, the model can't learn anything meaningful from it, and it complicates encoding later — so we drop that single row.

In [6]:
df = df.drop(columns=['id'])

print('Gender value counts before:')
print(df['gender'].value_counts())

df = df[df['gender'] != 'Other'].reset_index(drop=True)

print('\nShape after cleanup:', df.shape)


Gender value counts before:
gender
Female    2994
Male      2115
Other        1
Name: count, dtype: int64

Shape after cleanup: (5109, 11)


## 3. Separate Features (X) and Target (y)

In [7]:
X = df.drop(columns=['stroke'])
y = df['stroke']

print('Features shape:', X.shape)
print('Target distribution:')
print(y.value_counts(normalize=True).round(3))


Features shape: (5109, 10)
Target distribution:
stroke
0    0.951
1    0.049
Name: proportion, dtype: float64


## 4. Train-Test Split — BEFORE Any Cleaning

We use `stratify=y` so both the train and test sets keep roughly the same stroke/no-stroke ratio as the original data. Without this, we could accidentally end up with a test set that has almost no stroke cases at all, which would make evaluation meaningless.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('\nTrain stroke rate:', y_train.mean().round(4))
print('Test stroke rate:', y_test.mean().round(4))


Train shape: (4087, 10)
Test shape: (1022, 10)

Train stroke rate: 0.0487
Test stroke rate: 0.0489


## 5. Build the Preprocessing Pipeline

We separate columns into two groups:
- **Numeric columns** (age, avg_glucose_level, bmi): fill missing values with the median, then scale them so they're on a similar range.
- **Categorical columns** (gender, ever_married, work_type, Residence_type, smoking_status): fill any missing values with the most frequent value, then one-hot encode them (turn each category into its own 0/1 column).

This is wrapped in a `ColumnTransformer` — a single object that does all of this consistently, and can be reused later on new patient data in the app.

In [9]:
numeric_features = ['age', 'avg_glucose_level', 'bmi']
categorical_features = ['gender', 'hypertension', 'heart_disease', 'ever_married',
                         'work_type', 'Residence_type', 'smoking_status']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


## 6. Fit on Training Data Only, Then Transform Both

This is the most important line in the notebook: `fit_transform` is only ever called on `X_train`. The test set only ever gets `transform` — it's never allowed to influence how the imputer or scaler learned its values. That's what prevents data leakage.

In [10]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print('Processed train shape:', X_train_processed.shape)
print('Processed test shape:', X_test_processed.shape)
print('\nNumber of features after encoding:', len(feature_names))


Processed train shape: (4087, 22)
Processed test shape: (1022, 22)

Number of features after encoding: 22


**Your observation:** _(Why did the number of columns increase compared to the original data?)_

> 

## 7. Fix Class Imbalance with SMOTE — Training Data Only

SMOTE creates synthetic examples of the minority class (stroke cases) so the model sees a more balanced dataset during training. We apply it **only to the training set, after encoding**. The test set stays exactly as it was — untouched, imbalanced, and realistic — because that's what real-world data actually looks like.

In [11]:
print('Before SMOTE:')
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)

print('\nAfter SMOTE:')
print(pd.Series(y_train_resampled).value_counts())


Before SMOTE:
stroke
0    3888
1     199
Name: count, dtype: int64

After SMOTE:
stroke
0    3888
1    3888
Name: count, dtype: int64


**Your observation:** _(Why is it wrong to apply SMOTE before the train-test split, or on the test set?)_

> 

## 8. Save Everything for Phase 4

We save:
- The fitted preprocessor (so the exact same cleaning steps can be reused later in the Streamlit app)
- The processed and balanced training data
- The processed (untouched) test data
- The feature names, useful later for interpreting the model

In [12]:
joblib.dump(preprocessor, '../models/preprocessor.pkl')

joblib.dump({
    'X_train': X_train_resampled,
    'y_train': y_train_resampled,
    'X_test': X_test_processed,
    'y_test': y_test,
    'feature_names': feature_names
}, '../data/processed/processed_data.pkl')

print('Saved preprocessor.pkl and processed_data.pkl')


Saved preprocessor.pkl and processed_data.pkl


## Summary

Write 2-3 sentences, in your own words, on what preprocessing decisions you made and why. This is presentation material.

- 
- 
- 